# Multicoil

| Paper | Organ | Acceleration factor | Mask type | Train set | Test set | Sens est | PSNR | SSIM |
|---|---|---:|---|---|---|---|---|---|
| GrappaNet | Knee | 4 | Random Cartesian | 973 vols | fastMRI validation split | Yes | 40.74 | 0.957 |
| GrappaNet | Knee | 8 | Random Cartesian | 973 vols | fastMRI validation split | Yes | 36.76 | 0.922 |
| MFFVN | Brain | 4 | 1D Gaussian Cartesian (8% center) | 90 vols (~960 slices) | validation split | Yes | 34.7 | 0.939 |
| Joint-ICNet | Brain | 4 | 1D Cartesian equispaced | ~20% fastMRI train | fastMRI test | Yes | 36.21–36.08 | 0.943–0.957 |
| Joint-ICNet | Brain | 8 | 1D Cartesian equispaced | ~20% fastMRI train | fastMRI test | Yes | 31.27–33.12 | 0.903–0.921 |
| SUNO | Knee | 4 | Learned scan-adaptive Cartesian | 1514 slices (~156 scans) | fastMRI test split | Yes | 31.74 | 0.940 |
| SUNO | Knee | 8 | Learned scan-adaptive Cartesian | 1514 slices (~156 scans) | fastMRI test split | Yes | 28.66 | 0.801 |
| **FI VarNet** | **Brain** | **4** | **Cartesian (8% center)** | **~5800 vols** | **fastMRI test (558 vols)** | **Yes (learned)** | **41.45** | **0.9596** |
| **FI VarNet** | **Brain** | **5** | **Cartesian (7% center)** | **~5800 vols** | **fastMRI test** | **Yes (learned)** | **40.46** | **0.9552** |
| **FI VarNet** | **Knee** | **4** | **Cartesian** | **973 vols** | **validation split** | **Yes (learned)** | **40.08** | **0.9236** |

In [ ]:
import re
import matplotlib.pyplot as plt
from collections import defaultdict

ANSI_RE = re.compile(r'\x1b\[[0-9;]*m')


def parse_log(log_path):
    """Parse a training log.txt and return (train, val_loss, val_metrics) dicts."""
    train = defaultdict(list)
    val_loss = defaultdict(list)
    val_metrics = defaultdict(list)

    train_re     = re.compile(r'step (\d+)/\d+')
    val_loss_re  = re.compile(r'\[val @ step (\d+)\]\s+loss=')
    val_metric_re = re.compile(r'\[val @ step (\d+)\]\s+PSNR=')
    kv_re        = re.compile(r'(\w+)=([\d.eE+\-]+)')

    with open(log_path) as f:
        for raw in f:
            line = ANSI_RE.sub('', raw).strip()
            line = re.sub(r'^\[.*?\]\[.*?\]\s*', '', line)

            m = train_re.match(line)
            if m:
                step = int(m.group(1))
                for key, val in kv_re.findall(line):
                    train[key].append((step, float(val)))
                continue

            m = val_loss_re.match(line)
            if m:
                step = int(m.group(1))
                for key, val in kv_re.findall(line):
                    val_loss[key].append((step, float(val)))
                continue

            m = val_metric_re.match(line)
            if m:
                step = int(m.group(1))
                for key, val in kv_re.findall(line):
                    val_metrics[key].append((step, float(val)))

    return train, val_loss, val_metrics


def _unzip(pairs):
    steps, vals = zip(*pairs)
    return list(steps), list(vals)


def plot_training_log(log_path, label=None):
    """Parse log_path and plot all training/validation metrics.

    Parameters
    ----------
    log_path : str
        Path to a log.txt produced by train_multicoil.py.
    label : str, optional
        Legend label prefix. Defaults to log_path.
    """
    label = label or log_path
    train, val_loss, val_metrics = parse_log(log_path)

    train_loss_keys = [k for k in train if k != 'lr']
    val_loss_keys   = list(val_loss.keys())
    val_metric_keys = list(val_metrics.keys())
    extra_val_keys  = [k for k in val_loss_keys if k not in train_loss_keys]

    fig_cols = max(len(train_loss_keys), len(val_metric_keys), 1)
    fig, axes = plt.subplots(3, fig_cols, figsize=(5 * fig_cols, 12), squeeze=False)
    fig.suptitle(label, fontsize=11)

    # Row 0 – train loss components (+ val overlay where key matches)
    for col, key in enumerate(train_loss_keys):
        ax = axes[0][col]
        if train[key]:
            sx, sy = _unzip(train[key])
            ax.plot(sx, sy, label=f'train {key}', alpha=0.8)
        if key in val_loss and val_loss[key]:
            vx, vy = _unzip(val_loss[key])
            ax.plot(vx, vy, 'o-', label=f'val {key}', markersize=4)
        ax.set_title(key)
        ax.set_xlabel('step')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    for col in range(len(train_loss_keys), fig_cols):
        axes[0][col].set_visible(False)

    # Row 1 – val-only loss keys not already shown, or LR
    for col, key in enumerate(extra_val_keys):
        ax = axes[1][col]
        vx, vy = _unzip(val_loss[key])
        ax.plot(vx, vy, 'o-', label=f'val {key}', markersize=4)
        ax.set_title(f'val {key}')
        ax.set_xlabel('step')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    if not extra_val_keys and train.get('lr'):
        ax = axes[1][0]
        sx, sy = _unzip(train['lr'])
        ax.plot(sx, sy, color='tab:orange')
        ax.set_title('learning rate')
        ax.set_xlabel('step')
        ax.grid(True, alpha=0.3)
    for col in range(max(len(extra_val_keys), 1), fig_cols):
        axes[1][col].set_visible(False)

    # Row 2 – val image-quality metrics
    for col, key in enumerate(val_metric_keys):
        ax = axes[2][col]
        vx, vy = _unzip(val_metrics[key])
        ax.plot(vx, vy, 'o-', markersize=4)
        ax.set_title(key)
        ax.set_xlabel('step')
        ax.grid(True, alpha=0.3)
    for col in range(len(val_metric_keys), fig_cols):
        axes[2][col].set_visible(False)

    plt.tight_layout()
    plt.show()


In [ ]:
# plot_training_log("results_multicoil/brain/x8/run3_8_coils/log.txt")
# plot_training_log("/fast/lev/TRUST-MRI/results_multicoil/brain/x8/run2_coil_decoders/log.txt", label="run2")
# plot_training_log("/fast/lev/TRUST-MRI/results_multicoil/brain/x8/run1/log.txt", label="run1")
plot_training_log("/fast/lev/TRUST-MRI/results_multicoil/brain/x8/run4_img_loss/log.txt")


# Oracle comparison


In [ ]:
import importlib, sys
sys.path.insert(0, '/fast/lev/TRUST-MRI')
import visualization_utils as vu
importlib.reload(vu)

RUN_DIR = '/fast/lev/TRUST-MRI/results/oracle_benchmark'
oracle  = vu.load_oracle_steps(RUN_DIR, dataset='brain', R=16)
entropy = vu.load_entropy_steps(RUN_DIR, dataset='brain', R=16)

print("Oracle sources :", list(oracle.keys()))
print("Entropy sources:", list(entropy.keys()))
for k, df in {**oracle, **entropy}.items():
    print(f"  {k}: {df['sample_id'].nunique()} slices, steps {df['step'].min()}–{df['step'].max()}")

In [ ]:
# PSNR comparison: Oracle (PSNR obj.), Oracle (SSFD obj.), LES, GEO
fig = vu.plot_psnr(oracle, entropy, dataset='Brain', R=16)
fig.savefig(RUN_DIR + '/oracle_psnr_comparison_brain_R16.png', dpi=150, bbox_inches='tight')
fig.show()

In [ ]:
# SSFD comparison: Oracle (PSNR obj.), Oracle (SSFD obj.), LES, GEO
# Requires LES/GEO runs with --compute_ssfd (via run_oracle_benchmark.sh)
fig = vu.plot_ssfd(oracle, entropy, dataset='Brain', R=16)
fig.savefig(RUN_DIR + '/oracle_ssfd_comparison_brain_R16.png', dpi=150, bbox_inches='tight')
fig.show()

# PS Posterior Sample Viewer

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
PS_RUN_DIR = '/fast/lev/TRUST-MRI/results/oracle_benchmark'
DATASET    = 'brain'
R          = 16
STEP       = 0   # acquisition step to visualise (0 = center-only mask)

# List available sample_ids that have a samples/ subdir
ps_base = pathlib.Path(PS_RUN_DIR) / 'entropy_acquisition' / f'dataset={DATASET}' / f'R={R}' / 'method=PS'
sample_ids = sorted(p.name.replace('sample_id=', '') for p in ps_base.glob('sample_id=*') if (p / 'samples').exists())
print(f"{len(sample_ids)} samples with saved posterior draws:")
for s in sample_ids:
    print(' ', s)


In [ ]:
SAMPLE_ID = sample_ids[0]   # change index or paste a sample_id string directly

samples_dir = ps_base / f'sample_id={SAMPLE_ID}' / 'samples'
step_str    = f'step{STEP:03d}'

# Load argmax and mean
argmax_path = samples_dir / f'{step_str}_argmax.npy'
mean_path   = samples_dir / f'{step_str}_mean.npy'
assert argmax_path.exists(), f"Not found: {argmax_path}  (did you run with --save_ps_samples?)"

argmax_img = np.load(argmax_path)
mean_img   = np.load(mean_path)

# Load all individual posterior samples (step{t}_sample{i}.npy)
ps_files = sorted(samples_dir.glob(f'{step_str}_sample*.npy'))
ps_imgs  = [np.load(p) for p in ps_files]
n_samples = len(ps_imgs)

# ── Normalise to [0,1] using GT-like range (max of argmax) ───────────────────
vmax = argmax_img.max()
vmin = 0.0

def show(ax, img, title):
    ax.imshow(img, cmap='gray', vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=9)
    ax.axis('off')

# ── Layout: row 0 = argmax + mean + std; rows 1+ = individual samples ────────
n_cols  = max(4, n_samples)
std_img = np.std(np.stack(ps_imgs, axis=0), axis=0)

fig, axes = plt.subplots(2, n_cols, figsize=(2.5 * n_cols, 5.5))
fig.suptitle(f'{SAMPLE_ID}  |  step={STEP}', fontsize=10)

show(axes[0, 0], argmax_img, 'Argmax')
show(axes[0, 1], mean_img,   f'PS mean (n={n_samples})')
axes[0, 2].imshow(std_img, cmap='hot', vmin=0)
axes[0, 2].set_title('PS std (hot)', fontsize=9)
axes[0, 2].axis('off')

# difference map: mean − argmax
diff = mean_img - argmax_img
vd = np.abs(diff).max()
axes[0, 3].imshow(diff, cmap='RdBu_r', vmin=-vd, vmax=vd)
axes[0, 3].set_title('mean − argmax', fontsize=9)
axes[0, 3].axis('off')
for col in range(4, n_cols):
    axes[0, col].axis('off')

# Individual posterior samples
for i, img in enumerate(ps_imgs):
    show(axes[1, i], img, f'sample {i}')
for col in range(n_samples, n_cols):
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()
print(f"std  — mean: {std_img.mean():.4f}  max: {std_img.max():.4f}")
print(f"|diff| mean: {np.abs(diff).mean():.4f}  max: {np.abs(diff).max():.4f}")


In [ ]:
import sys, pathlib
sys.path.insert(0, '/fast/lev/TRUST-MRI')
import torch
import numpy as np
import matplotlib.pyplot as plt

from experiments.experiment1.utils.model_loader import load_singlecoil_model, load_meditok
from experiments.experiment1.run_normal_trajectory import load_slice, make_center_mask
from autoregressive.models.generate import reconstruct_far, data_consistency
from autoregressive.sample.adaptive_sampling import posterior_sampling_scores
from data import transforms
import yaml

# ── Config ────────────────────────────────────────────────────────────────────
CONFIG     = '/fast/lev/TRUST-MRI/experiments/experiment1/config.yaml'
PS_RUN_DIR = '/fast/lev/TRUST-MRI/results/ps_debug'
DATASET    = 'brain'
R          = 16
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CONFIGS = [
    dict(label='T=1  topk=100',   temperature=1.0, top_k=100, top_p=1.0),
    dict(label='T=1  pure',       temperature=1.0, top_k=0,   top_p=1.0),
    dict(label='T=2  pure',       temperature=2.0, top_k=0,   top_p=1.0),
    dict(label='T=3  pure',       temperature=3.0, top_k=0,   top_p=1.0),
    dict(label='T=1  p=0.95',     temperature=1.0, top_k=0,   top_p=0.95),
    dict(label='T=2  p=0.95',     temperature=2.0, top_k=0,   top_p=0.95),
]
N_SAMPLES = 8
STEP      = 0

ps_base    = pathlib.Path(PS_RUN_DIR) / 'entropy_acquisition' / f'dataset={DATASET}' / f'R={R}' / 'method=PS'
sample_ids = sorted(p.name.replace('sample_id=', '') for p in ps_base.glob('sample_id=*') if (p / 'samples').exists())
assert sample_ids, f"No saved PS samples found under {ps_base}."
SAMPLE_ID  = sample_ids[0]
print(f"Using: {SAMPLE_ID}  ({len(sample_ids)} available)")

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

root = pathlib.Path('/fast/lev/TRUST-MRI')
model, ckpt_args = load_singlecoil_model(root / cfg['checkpoints'][DATASET][R], DEVICE)
tokenizer = load_meditok(root / cfg['meditok_config'], root / cfg['meditok_ckpt'], DEVICE)

data_root       = pathlib.Path(root / cfg['data_paths'][DATASET])
center_fraction = cfg.get('center_fraction', 0.04)
FAR_KW = dict(n_steps=1, far_mask_mode='cartesian', active_sampling_mask_type='cartesian',
              center_fractions=center_fraction)

parts       = SAMPLE_ID.split('_slice')
scan_stem   = parts[0].replace(f'{DATASET}_file_', '')
slice_index = int(parts[1])
h5_path     = sorted(data_root.glob(f'*{scan_stem}*'))[0]

resolution = getattr(ckpt_args, 'image_size', 256)
slice_data  = load_slice(h5_path, slice_index, resolution)
gt_c_image  = slice_data['gt_c_image'].to(DEVICE)
_, H, W, _  = gt_c_image.shape
gt_rss      = transforms.complex_abs(gt_c_image[0]).cpu().numpy()

center_mask = make_center_mask(W, center_fraction, DEVICE)
sample_mask = center_mask.clone()
if STEP > 0:
    ps_kw_sel = dict(temperature=1.0, top_k=100, top_p=1.0)
    for _ in range(STEP):
        mi = data_consistency(gt_c_image, torch.zeros_like(gt_c_image), sample_mask)
        recs = []
        for __ in range(N_SAMPLES):
            with torch.no_grad():
                rs, _ = reconstruct_far(model, tokenizer, mi, sample_mask,
                                        sampling_kwargs=ps_kw_sel, **FAR_KW)
            recs.append(transforms.complex_abs(rs[0]))
        sel_cols, _ = posterior_sampling_scores(torch.stack(recs), sample_mask, rank=1)
        sample_mask[0, 0, sel_cols, 0] = 1.0

masked_image = data_consistency(gt_c_image, torch.zeros_like(gt_c_image), sample_mask)

def calc_psnr(img, gt):
    mse = np.mean((img - gt) ** 2)
    return float(10 * np.log10(gt.max() ** 2 / mse)) if mse > 0 else float('inf')

# ── Plot ──────────────────────────────────────────────────────────────────────
vmax  = gt_rss.max()
n_cfg = len(CONFIGS)
fig, axes = plt.subplots(n_cfg + 1, N_SAMPLES + 2, figsize=(2.2 * (N_SAMPLES + 2), 2.5 * (n_cfg + 1)))
fig.suptitle(f'{SAMPLE_ID}  |  step={STEP}', fontsize=10)

def _show(ax, img, title='', cmap='gray'):
    ax.imshow(img, cmap=cmap, vmin=0, vmax=vmax)
    ax.set_title(title, fontsize=7)
    ax.axis('off')

_show(axes[0, 0], gt_rss, 'GT')
for col in range(1, N_SAMPLES + 2):
    axes[0, col].axis('off')

for row, cfg_entry in enumerate(CONFIGS, start=1):
    label = cfg_entry['label']
    sk    = {k: v for k, v in cfg_entry.items() if k != 'label'}
    imgs  = []
    for _ in range(N_SAMPLES):
        with torch.no_grad():
            rec, _ = reconstruct_far(model, tokenizer, masked_image, sample_mask,
                                     sampling_kwargs=sk, **FAR_KW)
        imgs.append(transforms.complex_abs(rec[0]).cpu().numpy())

    mean_img  = np.mean(imgs, axis=0)
    std_img   = np.std(imgs,  axis=0)
    mean_psnr = calc_psnr(mean_img, gt_rss)

    _show(axes[row, 0], mean_img, f'{label}\nmean (PSNR={mean_psnr:.2f})')
    vd = std_img.max() or 1e-6
    axes[row, 1].imshow(std_img, cmap='hot', vmin=0, vmax=vd)
    axes[row, 1].set_title(f'std  μ={std_img.mean():.3f}', fontsize=7)
    axes[row, 1].axis('off')
    for i, img in enumerate(imgs):
        _show(axes[row, i + 2], img, f's{i} ({calc_psnr(img, gt_rss):.2f})')

    print(f"{label:20s}  mean PSNR={mean_psnr:.2f}  std mean={std_img.mean():.4f}")

plt.tight_layout()
plt.show()


In [ ]:
# PS config sweep — PSNR vs acquisition step
# Runs the full B-step acquisition loop for each config on one slice.
# Requires model/tokenizer/gt_c_image/gt_rss/FAR_KW/calc_psnr from the cell above.

PS_SWEEP_CONFIGS = [
    dict(label='T=1  topk=100',  temperature=1.0, top_k=100, top_p=1.0),  # original default
    dict(label='T=1  pure',      temperature=1.0, top_k=0,   top_p=1.0),
    dict(label='T=2  pure',      temperature=2.0, top_k=0,   top_p=1.0),
    dict(label='T=1  p=0.95',    temperature=1.0, top_k=0,   top_p=0.95),
]
SWEEP_N_SAMPLES = 16   # more samples → more stable variance estimate for line selection

B_budget = max(1, W // R - int(W * center_fraction))
steps    = list(range(B_budget + 1))
print(f"Budget: {B_budget} steps  |  n_samples per step: {SWEEP_N_SAMPLES}")

def run_ps_acquisition(sk, n_samples):
    mask   = make_center_mask(W, center_fraction, DEVICE)
    psnrs  = []
    for t in range(B_budget + 1):
        mi = data_consistency(gt_c_image, torch.zeros_like(gt_c_image), mask)
        with torch.no_grad():
            rec, _ = reconstruct_far(model, tokenizer, mi, mask, **FAR_KW)
        psnrs.append(calc_psnr(transforms.complex_abs(rec[0]).cpu().numpy(), gt_rss))

        if t < B_budget:
            recs = []
            for _ in range(n_samples):
                with torch.no_grad():
                    rs, _ = reconstruct_far(model, tokenizer, mi, mask,
                                            sampling_kwargs=sk, **FAR_KW)
                recs.append(transforms.complex_abs(rs[0]))
            sel_cols, _ = posterior_sampling_scores(torch.stack(recs), mask, rank=1)
            mask[0, 0, sel_cols, 0] = 1.0
    return psnrs

fig, ax = plt.subplots(figsize=(7, 4))

for cfg_entry in PS_SWEEP_CONFIGS:
    label = cfg_entry['label']
    sk    = {k: v for k, v in cfg_entry.items() if k != 'label'}
    print(f"  {label} ...", end=' ', flush=True)
    psnrs = run_ps_acquisition(sk, SWEEP_N_SAMPLES)
    ax.plot(steps, psnrs, marker='o', markersize=4, label=f"{label}  (final={psnrs[-1]:.2f})")
    print(f"step0={psnrs[0]:.2f}  final={psnrs[-1]:.2f}  gain={psnrs[-1]-psnrs[0]:.2f}")

ax.set_xlabel('Acquisition step')
ax.set_ylabel('PSNR (dB)')
ax.set_title(f'PS config sweep — {SAMPLE_ID}\n(argmax recon PSNR, n_samples={SWEEP_N_SAMPLES})')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# PS config sweep — PSNR vs step (multi-slice, from saved parquets)
# Run the sweep first:
#   bash experiments/experiment1/run_oracle_benchmark.sh \
#     --run_dir results/ps_sweep \
#     --datasets brain --acceleration 16 \
#     --stratified --scans_per_contrast 3 --center_slices 5 \
#     --ps_sweep --gpus 4

import pandas as pd
import pathlib
import numpy as np
import matplotlib.pyplot as plt

PS_SWEEP_RUN_DIR = '/fast/lev/TRUST-MRI/results/ps_sweep'
DATASET = 'brain'
R       = 16
TAGS    = ['PS_T1_k100', 'PS_T1_pure', 'PS_T2_pure', 'PS_T1_p95']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for metric, ax in zip(['PSNR', 'PSNR_ps_avg'], axes):
    n_slices = None
    for tag in TAGS:
        p = (pathlib.Path(PS_SWEEP_RUN_DIR)
             / 'entropy_acquisition' / f'dataset={DATASET}' / f'R={R}'
             / f'method={tag}' / 'metadata.parquet')
        if not p.exists():
            print(f'  missing: {p}')
            continue
        df = pd.read_parquet(p)
        grp = df.groupby('step')[metric]
        mean, se = grp.mean(), grp.sem()
        steps = mean.index.tolist()
        ax.plot(steps, mean.values, marker='o', markersize=4, label=tag)
        ax.fill_between(steps, (mean - se).values, (mean + se).values, alpha=0.15)
        n_slices = df['scan_id'].nunique()
    ylabel = 'PSNR argmax (dB)' if metric == 'PSNR' else 'PSNR PS-avg (dB)'
    title_suffix = f' ({n_slices} slices)' if n_slices else ''
    ax.set_xlabel('Acquisition step')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{DATASET} R={R} — {ylabel}{title_suffix}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# Entropy Validation

In [ ]:
import importlib, sys
sys.path.insert(0, '/fast/lev/TRUST-MRI')
import visualization_utils as vu
importlib.reload(vu)

RUN_DIR = '/fast/lev/TRUST-MRI/results/oracle_benchmark'

In [ ]:
# Experiment A — Patch error vs entropy quintile (random masks)
# Reads: results/oracle_benchmark/random_mask_diagnostic/dataset=.../R=.../metadata.parquet
import pandas as pd, numpy as np

# Spearman ρ summary
print("Spearman ρ (patch entropy vs patch MSE, random masks):")
print(f"{'Dataset':>10}  {'R':>4}  {'mean ρ':>8}  {'std ρ':>7}")
for ds, R in [('brain', 8), ('brain', 16), ('knee', 8), ('knee', 16)]:
    df = pd.read_parquet(f'{RUN_DIR}/random_mask_diagnostic/dataset={ds}/R={R}/metadata.parquet')
    print(f"{ds:>10}  {R:>4}  {df['spearman_rho'].mean():>8.3f}  {df['spearman_rho'].std():>7.3f}")

print()
fig_brain = vu.plot_entropy_bins(RUN_DIR, dataset='brain', accelerations=[8, 16])
fig_brain.show()

fig_knee = vu.plot_entropy_bins(RUN_DIR, dataset='knee', accelerations=[8, 16])
fig_knee.show()

In [ ]:
# Experiment B — Normalized entropy + MSE vs acquisition progress (τ = t/B)
# Reads: results/oracle_benchmark/entropy_acquisition/dataset=.../R=.../method=.../metadata.parquet
fig_acq_brain = vu.plot_acquisition_curves(RUN_DIR, dataset='brain', accelerations=[8, 16])
fig_acq_brain.show()

fig_acq_knee = vu.plot_acquisition_curves(RUN_DIR, dataset='knee', accelerations=[8, 16])
fig_acq_knee.show()

# Patch Entropy Investigation

In [ ]:
import importlib, sys
sys.path.insert(0, '/fast/lev/TRUST-MRI')
import visualization_utils as vu
importlib.reload(vu)

PE_DIR   = '/fast/lev/TRUST-MRI/results/gt_recon/'
DATASET  = 'brain'   # 'brain' or 'knee'
R        = 8        # 8 or 16
METHOD   = 'GT_US'     # 'LES' or 'GEO'

samples = vu.list_patch_entropy_samples(PE_DIR, DATASET, R, METHOD)
print(f"{len(samples)} completed samples")
print('\n'.join(samples[:100]))

In [ ]:
# Load one sample — change sample_id to explore different slices
sample_id = samples[51]
data = vu.load_patch_entropy_data(PE_DIR, DATASET, R, METHOD, sample_id)
print(f"Loaded: B={data['B']} steps, grid={data['H_p']}×{data['W_p']}, K={data['K']} codebooks")
print(f"GT RSS shape: {data['gt_rss'].shape}")

# --- entropy_sel controls which dimensions to sum ---
# None                         → sum all channels + codebooks  (default)
# {'channel': 0}               → real channel only
# {'channel': 1}               → imag channel only
# {'codebook': 3}              → codebook 3, both channels summed
# {'channel': 0, 'codebook': 3}→ codebook 3, real channel only
SEL = None   # change this; all plots below use SEL
SEL = {'codebooks': list(range(1,8))}

# Reference grid: find patch (row, col) for single-patch plots
fig = vu.show_patch_reference_grid(data)

In [ ]:
# Viz 1 — Entropy of a specific patch over acquisition steps
# Set patch_row, patch_col using the reference grid above
fig = vu.plot_patch_entropy_over_steps(data, patch_row=11, patch_col=10, entropy_sel=SEL)
fig = vu.plot_patch_entropy_over_steps(data, patch_row=9, patch_col=4, entropy_sel=SEL)

In [ ]:
# Viz 2 — Selection progression: which patch / k-space line was chosen at each step
fig = vu.plot_selection_progression(data, entropy_sel=SEL)

In [ ]:
# Viz 3 — Entropy reduction heatmap: how much did each patch's entropy drop?
# relative=True → normalized by step-0 entropy; relative=False → absolute nats
# delta = H0 - H
fig = vu.plot_entropy_reduction_grid(data, relative=True, entropy_sel=SEL)

In [ ]:
# Viz 4 — Entropy of the argmax-entropy patch at each step vs global mean entropy
fig = vu.plot_selected_patch_entropy(data, entropy_sel=SEL)

In [ ]:
# Viz 5 — Entropy map sequence: all B+1 heatmaps side by side
# Shows WHERE entropy is high and how it evolves step by step
fig = vu.plot_entropy_map_sequence(data, entropy_sel=SEL)

In [ ]:
# Viz 6 — Per-codebook / per-channel entropy heatmaps at a specific step
# entropy_sel examples:
#   None                          → all 2×8=16 maps shown (rows=channels, cols=codebooks)
#   {'channel': 0}                → real channel only (8 maps across 1 row)
#   {'channel': 1}                → imag channel only
#   {'codebook': 3}               → codebook 3, both channels shown
#   {'channel': 0, 'codebook': 3} → single map: codebook 3, real channel
fig = vu.plot_per_codebook_entropy(data, step=0, entropy_sel=SEL)
fig = vu.plot_per_codebook_entropy(data, step=10, entropy_sel=SEL)
fig = vu.plot_per_codebook_entropy(data, step=20, entropy_sel=SEL)

In [ ]:
# Viz 7 — Entropy delta per step: how much did each patch change at each acquisition step?
# B heatmaps of (entropy_t - entropy_{t-1}); blue=decrease, red=increase
fig = vu.plot_entropy_delta_sequence(data, entropy_sel=SEL)

In [ ]:
# Viz 8 — Entropy vs patch MSE scatter (validates entropy as uncertainty signal)
# Spearman ρ shown in legend; change step= to check different acquisition steps
fig = vu.plot_entropy_mse_scatter(data, step=0, entropy_sel=SEL)

In [ ]:
# Viz 9 — Global mean entropy vs acquisition step
# Confirms whether entropy decreases at all across the acquisition budget
fig = vu.plot_mean_entropy_vs_coverage(data, entropy_sel=SEL)

In [ ]:
# KL divergence from step-0 baseline: does acquiring k-space lines shift any patch distributions?
# High KL = distribution moved a lot relative to center-only reconstruction.
# Requires re-running run_patch_entropy_experiment.py (saves kl_from_t0_t*.npy).
fig = vu.plot_kl_map_sequence(data, entropy_sel=SEL)

In [ ]:
# PSNR vs acquisition step — does reconstruction quality actually improve?
fig = vu.plot_psnr_vs_step(data)

In [ ]:
# Bar chart: mean entropy per codebook at a given step
# cb0 has no AR conditioning → expect higher entropy than cb1-7
fig = vu.plot_entropy_per_codebook_bar(data, step=0)

# To use only codebooks 1-7 in any other viz, pass:
# entropy_sel = {'codebooks': list(range(1, 8))}

In [ ]:
# Bar chart: mean entropy per codebook aggregated over ALL samples in the dataset
# Error bars = std across samples. step=0 = initial (center-only) reconstruction.
for i in range(0, 20, 1):
    fig = vu.plot_entropy_per_codebook_bar_dataset(PE_DIR, DATASET, R, METHOD, step=i)

In [ ]:
# GT vs reconstruction entropy — heatmaps for one slice
# Requires: run run_gt_entropy.py first (see below)
# Compares: GT (full k-space) | step-0 (center only) | step-B (final acquisition)
fig = vu.plot_gt_comparison_heatmaps(data, entropy_sel=SEL)

In [ ]:
# GT vs step-0 vs step-B per-codebook bar chart, aggregated over all samples
fig = vu.plot_gt_vs_recon_bar_dataset(PE_DIR, DATASET, R, METHOD)

# Token Accuracy

In [ ]:
# Run first: bash experiments/experiment1/run_token_accuracy.sh
#
# top-k accuracy: fraction of patches where GT token is in model's top-k predictions
# NLL: -log p(gt_token) averaged over all patches/channels/codebooks
# RANDOM: fixed random mask at each budget level (seed=42), not adaptive

TOKEN_ACC_DIR = '/fast/lev/TRUST-MRI/results/token_accuracy'

# Brain R=8
fig = vu.plot_token_accuracy(TOKEN_ACC_DIR, dataset='brain', R=8)

In [ ]:
# Change dataset/R to explore other settings
# fig = vu.plot_token_accuracy(TOKEN_ACC_DIR, dataset='brain', R=16)
# fig = vu.plot_token_accuracy(TOKEN_ACC_DIR, dataset='knee',  R=8)
# fig = vu.plot_token_accuracy(TOKEN_ACC_DIR, dataset='knee',  R=16)

# Show only a subset of k values or methods:
# fig = vu.plot_token_accuracy(TOKEN_ACC_DIR, 'brain', 8, methods=['LES_ZM', 'RANDOM'], ks=[1, 10])
fig = vu.plot_token_accuracy(TOKEN_ACC_DIR, dataset='brain', R=16)

In [ ]:
# GT reconstruction experiment — image quality + entropy
# Run first: bash experiments/experiment1/run_gt_recon_experiment.sh
#
# To use existing entropy cells (Viz 1-9) with GT recon data, set:
#   PE_DIR  = '/fast/lev/TRUST-MRI/results/gt_recon'
#   METHOD  = 'GT'
# then re-run the setup cell and reload data.

GT_PE_DIR = '/fast/lev/TRUST-MRI/results/gt_recon'
gt_samples = vu.list_patch_entropy_samples(GT_PE_DIR, DATASET, R, 'GT')
print(f"{len(gt_samples)} GT recon samples available")

gt_data = vu.load_patch_entropy_data(GT_PE_DIR, DATASET, R, 'GT', gt_samples[0])
fig = vu.plot_gt_recon_comparison(gt_data)

In [ ]:
import json, pathlib, numpy as np, pandas as pd
from skimage.metrics import structural_similarity as ssim_fn

def summarize_gt_recon_metrics(run_dir, datasets=('brain', 'knee'), Rs=(8, 16), method='GT'):
    """Compute metrics per scan-volume, then average across scans.

    PSNR and NMSE are computed over the full 3D volume (all slices stacked).
    SSIM is averaged per slice then averaged per scan (no standard 3D formulation).
    SSFD is averaged per slice then averaged per scan.
    """
    base = pathlib.Path(run_dir) / 'patch_entropy'
    all_scan_rows = []

    for ds in datasets:
        for R in Rs:
            method_dir = base / f'dataset={ds}' / f'R={R}' / f'method={method}'
            if not method_dir.exists():
                continue

            # Group sample dirs by scan_id
            scan_slices = {}
            for done_file in method_dir.glob('sample_id=*/done.json'):
                meta = json.load(open(done_file))[0]
                scan_id = meta['scan_id']
                sdir = done_file.parent
                if scan_id not in scan_slices:
                    scan_slices[scan_id] = []
                scan_slices[scan_id].append((int(meta['slice_id']), sdir, meta))

            for scan_id, slices in scan_slices.items():
                slices.sort(key=lambda x: x[0])  # sort by slice index

                rec_vol, gt_vol, slice_ssims, slice_ssfds = [], [], [], []
                for _, sdir, meta in slices:
                    rec = np.load(str(sdir / 'rec_rss.npy'))
                    gt  = np.load(str(sdir / 'gt_rss.npy'))
                    rec_vol.append(rec)
                    gt_vol.append(gt)
                    # SSIM per slice (data_range from GT)
                    s = ssim_fn(rec, gt, data_range=gt.max() - gt.min())
                    slice_ssims.append(s)
                    slice_ssfds.append(meta.get('SSFD', float('nan')))

                rec_vol = np.stack(rec_vol)   # (S, H, W)
                gt_vol  = np.stack(gt_vol)    # (S, H, W)

                # Volume-level PSNR and NMSE
                max_val = gt_vol.max()
                mse     = np.mean((rec_vol - gt_vol) ** 2)
                psnr    = float(10 * np.log10(max_val ** 2 / mse)) if mse > 0 else float('inf')
                nmse    = float(np.sum((rec_vol - gt_vol) ** 2) / np.sum(gt_vol ** 2))

                all_scan_rows.append({
                    'dataset': ds, 'R': R, 'scan_id': scan_id,
                    'PSNR': psnr,
                    'SSIM': float(np.mean(slice_ssims)),
                    'NMSE': nmse,
                    'SSFD': float(np.nanmean(slice_ssfds)),
                })

    df = pd.DataFrame(all_scan_rows)
    summary = (df.groupby(['dataset', 'R'])[['PSNR', 'SSIM', 'NMSE', 'SSFD']]
                 .agg(['mean', 'std'])
                 .round(4))
    summary.columns = [f'{m}_{s}' for m, s in summary.columns]
    return summary

GT_RUN_DIR = '/fast/lev/TRUST-MRI/results/gt_recon'
summary = summarize_gt_recon_metrics(GT_RUN_DIR)
# print(summary.to_string())
# print()
print(summary.to_markdown())

# Foreground Masks

In [ ]:
"""
Visualise GLES foreground masks alongside GT images.

Masks are generated by generate_foreground_masks.py (edge method):
  - Column-energy profile smoothed and differentiated horizontally
  - First & last strong edge give left/right brain boundary
  - All patch columns in [left, right] are foreground
  → single connected strip, no holes

Each row shows one (scan, slice) sample:
  col 0 — GT magnitude image
  col 1 — foreground mask  (white = foreground patch, patch grid shown)
  col 2 — green overlay on GT image

Sampling:
  n_slices_per_scan = max(1, N // total_unique_scans)
  Scans picked WITH repetition; slices picked WITHOUT repetition within each scan.
"""

import pathlib, random, math
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Config ────────────────────────────────────────────────────────────────────
FG_DIR     = pathlib.Path('/fast/lev/TRUST-MRI/fastmri/brain/singlecoil_emulated_val_fg_masks')
GT_DIR     = pathlib.Path('/fast/lev/TRUST-MRI/fastmri/brain/singlecoil_emulated_val')
N          = 20       # total (scan, slice) samples to display
SEED       = 42
H_P        = 16       # must match --H_p used when generating masks
RESOLUTION = 256

# ── Helpers ───────────────────────────────────────────────────────────────────
def _ifft2(kspace):
    shifted = np.fft.ifftshift(kspace, axes=(-2, -1))
    img     = np.fft.ifftn(shifted, axes=(-2, -1), norm="ortho")
    return np.fft.fftshift(img, axes=(-2, -1))

def load_gt_rss(h5_path, slice_idx):
    with h5py.File(h5_path, 'r') as f:
        kspace = f['kspace'][slice_idx]
    c = _ifft2(kspace)
    img = np.stack([c.real, c.imag], axis=-1).astype(np.float32)
    H, W, _ = img.shape
    if H < RESOLUTION or W < RESOLUTION:
        ph, pw = max(0, RESOLUTION-H), max(0, RESOLUTION-W)
        img = np.pad(img, ((ph//2, ph-ph//2), (pw//2, pw-pw//2), (0,0)))
    r0, c0 = (img.shape[0]-RESOLUTION)//2, (img.shape[1]-RESOLUTION)//2
    img = img[r0:r0+RESOLUTION, c0:c0+RESOLUTION]
    mag = np.sqrt(img[...,0]**2 + img[...,1]**2)
    mv = mag.max()
    return (mag / mv) if mv > 0 else mag

def upsample_mask(fg_mask):
    ph = RESOLUTION // H_P
    return fg_mask.repeat(ph, axis=0).repeat(ph, axis=1)

# ── Sampling ──────────────────────────────────────────────────────────────────
rng = random.Random(SEED)

npz_files = sorted(FG_DIR.glob('*.npz'))
assert npz_files, f"No .npz files found in {FG_DIR} — run generate_foreground_masks.py first"

total_unique_scans = len(npz_files)
n_slices_per_scan  = max(1, N // total_unique_scans)
n_scans_needed     = math.ceil(N / n_slices_per_scan)

selected_npz = rng.choices(npz_files, k=n_scans_needed)

samples = []
for npz_path in selected_npz:
    if len(samples) >= N:
        break
    data = np.load(npz_path)
    n_avail = data['fg_masks'].shape[0]
    idxs = rng.sample(range(n_avail), min(n_slices_per_scan, n_avail))
    for idx in idxs:
        samples.append((npz_path, idx))
        if len(samples) >= N:
            break

print(f"{len(npz_files)} scan NPZ files  |  n_slices_per_scan={n_slices_per_scan}  |  showing {len(samples)} samples")

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(len(samples), 3, figsize=(10, 3.2 * len(samples)),
                         gridspec_kw={'wspace': 0.04, 'hspace': 0.35})
if len(samples) == 1:
    axes = axes[np.newaxis, :]

for col, title in enumerate(['GT magnitude', 'Foreground mask', 'Mask overlay']):
    axes[0, col].set_title(title, fontsize=11, pad=6)

for row, (npz_path, slice_idx) in enumerate(samples):
    scan_stem = npz_path.stem
    h5_path   = GT_DIR / (scan_stem + '.h5')
    data      = np.load(npz_path)
    fg_mask   = data['fg_masks'][slice_idx]   # (H_p, H_p)
    fg_full   = upsample_mask(fg_mask)        # (256, 256)
    gt_rss    = load_gt_rss(h5_path, slice_idx)

    n_fg  = int(fg_mask.sum())
    n_tot = fg_mask.size

    # Col 0 — GT
    axes[row, 0].imshow(gt_rss, cmap='gray', vmin=0, vmax=1, interpolation='none')
    axes[row, 0].set_ylabel(f"{scan_stem[-20:]}\nslice {slice_idx}", fontsize=7, labelpad=3)

    # Col 1 — binary mask with patch grid lines
    axes[row, 1].imshow(fg_full, cmap='gray', vmin=0, vmax=1, interpolation='none')
    step = RESOLUTION // H_P
    for k in range(0, RESOLUTION, step):
        axes[row, 1].axhline(k - 0.5, color='gray', lw=0.3, alpha=0.5)
        axes[row, 1].axvline(k - 0.5, color='gray', lw=0.3, alpha=0.5)
    axes[row, 1].set_title(f"{n_fg}/{n_tot} fg patches", fontsize=8, pad=3)

    # Col 2 — green overlay
    ov = np.stack([gt_rss]*3, axis=-1)
    fg_bool = fg_full.astype(bool)
    ov[fg_bool, 0] = np.clip(ov[fg_bool, 0] * 0.5,       0, 1)
    ov[fg_bool, 1] = np.clip(ov[fg_bool, 1] * 0.5 + 0.5, 0, 1)
    ov[fg_bool, 2] = np.clip(ov[fg_bool, 2] * 0.5,       0, 1)
    axes[row, 2].imshow(ov, interpolation='none')

    for ax in axes[row]:
        ax.axis('off')

fg_p = mpatches.Patch(color=(0, 0.5, 0), alpha=0.8, label='foreground')
bg_p = mpatches.Patch(color='black',      alpha=0.6, label='background')
fig.legend(handles=[fg_p, bg_p], loc='lower center', ncol=2,
           fontsize=9, frameon=False, bbox_to_anchor=(0.5, 0.0))
plt.suptitle(
    f"GLES foreground masks — edge method  |  {H_P}×{H_P} patch grid  |  edge_fraction=0.10",
    fontsize=12, y=1.002,
)
plt.show()


# Reconstruction Visualization

In [ ]:
"""
Reconstruction Visualization

Compares reconstructed slices against ground-truth for any recon/ directory.
Recon H5 files must have key 'reconstruction' with shape (N_slices, H, W).
Both GT and recon are in original (unnormalized) scale.

Config:
  RECON_DIR  — directory of recon .h5 files
  GT_DIR     — directory of original fastMRI .h5 files (for ground truth)
  SCAN_NAME  — exact filename (e.g. 'file_brain_AXT2_210_2100027.h5') to inspect,
               or None to randomly sample N slices across all available scans
  N          — slices to show when SCAN_NAME is None
"""

import pathlib, random
import h5py
import numpy as np
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
RECON_DIR = pathlib.Path('/fast/lev/TRUST-MRI/results/oracle_benchmark/entropy_acquisition/dataset=brain/R=16/method=TDS/recon')
GT_DIR    = pathlib.Path('/fast/lev/TRUST-MRI/fastmri/brain/singlecoil_emulated_val')
SCAN_NAME = None   # e.g. 'file_brain_AXT2_210_2100027.h5' — or None for random
N         = 8      # slices to show when SCAN_NAME is None
SEED      = 42

# ── Helpers ───────────────────────────────────────────────────────────────────
def _load_gt_slice(gt_dir, scan_filename, slice_idx, resolution=256):
    """GT slice → (H, W) float32 magnitude in original (unnormalized) scale."""
    with h5py.File(pathlib.Path(gt_dir) / scan_filename, 'r') as f:
        kspace = f['kspace'][slice_idx]
    img_c = np.fft.fftshift(
        np.fft.ifftn(np.fft.ifftshift(kspace, axes=(-2,-1)), axes=(-2,-1), norm='ortho'),
        axes=(-2,-1),
    )
    img = np.stack([img_c.real, img_c.imag], axis=-1).astype(np.float32)
    H, W, _ = img.shape
    if H < resolution or W < resolution:
        ph, pw = max(0, resolution-H), max(0, resolution-W)
        img = np.pad(img, ((ph//2, ph-ph//2), (pw//2, pw-pw//2), (0, 0)))
    r0, c0 = (img.shape[0]-resolution)//2, (img.shape[1]-resolution)//2
    img = img[r0:r0+resolution, c0:c0+resolution]
    return np.sqrt(img[..., 0]**2 + img[..., 1]**2)   # original scale, no max_val division

def _psnr(pred, gt):
    mse  = np.mean((pred - gt) ** 2)
    vmax = float(gt.max())
    return float(10 * np.log10(vmax**2 / mse)) if mse > 0 and vmax > 0 else float('inf')

def _ssim_simple(pred, gt):
    """Global SSIM with data-range-scaled constants."""
    dr = float(gt.max())
    mu_x, mu_y = pred.mean(), gt.mean()
    sig_x  = pred.std();  sig_y = gt.std()
    sig_xy = ((pred - mu_x) * (gt - mu_y)).mean()
    c1, c2 = (0.01 * dr)**2, (0.03 * dr)**2
    return float(((2*mu_x*mu_y + c1) * (2*sig_xy + c2)) /
                 ((mu_x**2 + mu_y**2 + c1) * (sig_x**2 + sig_y**2 + c2)))

# ── List available scans ──────────────────────────────────────────────────────
recon_files = sorted(pathlib.Path(RECON_DIR).glob('*.h5'))
assert recon_files, f"No .h5 files found in {RECON_DIR}"

W = 65
print('─' * W)
print(f"  {str(RECON_DIR)}")
print('─' * W)
print(f"  {'Scan':<43}  {'slices':>6}  {'shape':>10}")
for rf in recon_files:
    with h5py.File(rf) as f:
        sh = f['reconstruction'].shape
    print(f"  {rf.name:<43}  {sh[0]:>6}  {sh[1]}×{sh[2]}")
print('─' * W)
print(f"\nSet SCAN_NAME='<name>.h5' to inspect one scan, or leave None to show {N} random slices.\n")

# ── Build sample list ─────────────────────────────────────────────────────────
rng = random.Random(SEED)

if SCAN_NAME is not None:
    target = pathlib.Path(RECON_DIR) / SCAN_NAME
    assert target.exists(), f"Not found: {target}"
    with h5py.File(target) as f:
        n_slices = f['reconstruction'].shape[0]
    samples = [(target, i) for i in range(n_slices)]
else:
    pool = []
    for rf in recon_files:
        with h5py.File(rf) as f:
            n = f['reconstruction'].shape[0]
        for i in range(n):
            pool.append((rf, i))
    samples = rng.sample(pool, min(N, len(pool)))

print(f"Showing {len(samples)} slice(s).")

# ── Plot ──────────────────────────────────────────────────────────────────────
gt_dir = pathlib.Path(GT_DIR)

fig, axes = plt.subplots(
    len(samples), 3,
    figsize=(12, 3.6 * len(samples)),
    gridspec_kw={'wspace': 0.06, 'hspace': 0.45},
)
if len(samples) == 1:
    axes = axes[np.newaxis, :]

for col, title in enumerate(['GT (full k-space)', 'Reconstruction', 'Δ = Recon − GT']):
    axes[0, col].set_title(title, fontsize=11, pad=6, fontweight='bold')

for row, (recon_path, slice_in_file) in enumerate(samples):
    with h5py.File(recon_path) as f:
        recon_vol = f['reconstruction'][()]          # (N_slices, H, W), original scale
    recon = recon_vol[slice_in_file].astype(np.float32)

    gt_path = gt_dir / recon_path.name
    if gt_path.exists():
        with h5py.File(gt_path) as f:
            n_gt = f['kspace'].shape[0]
        if n_gt == recon_vol.shape[0]:
            gt_idx, gt_note = slice_in_file, ''
        else:
            gt_idx = round(slice_in_file * (n_gt - 1) / max(recon_vol.shape[0] - 1, 1))
            gt_note = '≈'
        gt = _load_gt_slice(gt_dir, recon_path.name, gt_idx)
        has_gt = True
    else:
        gt = np.zeros_like(recon)
        gt_idx, gt_note, has_gt = slice_in_file, '?', False

    delta = recon - gt
    vmax  = max(float(gt.max()), 1e-8)
    vd    = max(float(np.abs(delta).max()), 1e-8)
    psnr  = _psnr(recon, gt) if has_gt else float('nan')
    ssim  = _ssim_simple(recon, gt) if has_gt else float('nan')

    def _annot(ax, text):
        ax.text(0.5, 0.02, text, transform=ax.transAxes, ha='center', va='bottom',
                fontsize=8, color='white',
                bbox=dict(facecolor='black', alpha=0.5, pad=1.5, boxstyle='round,pad=0.2'))

    axes[row, 0].imshow(gt, cmap='gray', vmin=0, vmax=vmax, interpolation='none')
    if not has_gt:
        axes[row, 0].text(0.5, 0.5, 'GT not found', transform=axes[row, 0].transAxes,
                          ha='center', va='center', color='red', fontsize=9)

    axes[row, 1].imshow(recon, cmap='gray', vmin=0, vmax=vmax, interpolation='none')
    _annot(axes[row, 1], f'PSNR {psnr:.1f} dB  |  SSIM {ssim:.3f}')

    im = axes[row, 2].imshow(delta, cmap='RdBu_r', vmin=-vd, vmax=vd, interpolation='none')
    _annot(axes[row, 2], f'max|Δ| = {vd:.4f}')
    plt.colorbar(im, ax=axes[row, 2], fraction=0.046, pad=0.04, format='%.4f')

    axes[row, 0].set_ylabel(
        f"{recon_path.stem[-22:]}\nrecon[{slice_in_file}] ↔ GT[{gt_note}{gt_idx}]",
        fontsize=7, labelpad=3,
    )
    for ax in axes[row]:
        ax.axis('off')

parent_name = '/'.join(str(RECON_DIR).split('/')[-3:])
plt.suptitle(f"Reconstruction vs GT  |  …/{parent_name}", fontsize=11, y=1.002)
plt.show()
